# SmolVLM Binary-Choice Fine-Tuning — Google Colab + A100 Ready

This notebook has been customized for **Google Colab** with an **A100 GPU**.

### Main changes from the Kaggle version
- All Kaggle paths converted to `/content/...`
- Added Kaggle API dataset download support
- Optimized for A100 with:
  - `bf16` mixed precision
  - TF32 enabled
  - Larger batch sizes
- Outputs saved under `/content/run_14_outputs`
- Optional Google Drive backup support added

### Recommended Runtime
- Runtime → Change runtime type
- Hardware accelerator → GPU
- GPU type → A100


In [ ]:
# Verify A100 is selected
import subprocess, torch

print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    total_gb = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  [{i}] {name}  ({total_gb:.1f} GB)")
    if "A100" not in name:
        print(f"  WARNING: device {i} is not an A100 — hyperparameters in this notebook are tuned for A100.")
        print("           Go to Runtime → Change runtime type → A100 GPU, then re-run.")

print()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


In [ ]:
# Install dependencies for Colab + A100
!pip uninstall -y -q torchao
!pip install -q -U transformers peft accelerate bitsandbytes kaggle scikit-learn pandas

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## Download the dataset

Kaggle auto-mounts competition data at `/kaggle/input/...`; Colab does not, so we fetch it via the Kaggle API.

**Option A — Kaggle API (default below):** upload your `kaggle.json` (Kaggle → Account → "Create New API Token"). You must have **accepted the competition rules** on the Kaggle competition page first, otherwise the download will 403.

**Option B — Google Drive:** if you've already pre-downloaded the data to Drive, skip the next cell, mount Drive, and copy/symlink it into `/content/pixels-to-predictions/`. The training script also searches `/content/drive/MyDrive/pixels-to-predictions`.


In [ ]:
# Option A: Kaggle API. Run this cell, pick your kaggle.json when prompted.
import os, shutil
from pathlib import Path
from google.colab import files

DATA_DIR = Path("/content/pixels-to-predictions")

if (DATA_DIR / "train.csv").exists() and (DATA_DIR / "val.csv").exists() and (DATA_DIR / "test.csv").exists():
    print(f"Dataset already present at {DATA_DIR} — skipping download.")
else:
    # 1. Get kaggle.json into ~/.kaggle/
    kaggle_dir = Path("/root/.kaggle")
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    if not (kaggle_dir / "kaggle.json").exists():
        print("Upload your kaggle.json (Kaggle → Account → 'Create New API Token').")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise RuntimeError("kaggle.json was not uploaded.")
        (kaggle_dir / "kaggle.json").write_bytes(uploaded["kaggle.json"])
    os.chmod(kaggle_dir / "kaggle.json", 0o600)

    # 2. Download + unzip the competition data into /content/pixels-to-predictions
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    !kaggle competitions download -c pixels-to-predictions -p {DATA_DIR}
    # If the line above 403s, accept the competition rules on Kaggle and re-run.
    !cd {DATA_DIR} && unzip -q -o pixels-to-predictions.zip && rm -f pixels-to-predictions.zip

# Sanity check
import os
for name in ("train.csv", "val.csv", "test.csv"):
    p = DATA_DIR / name
    print(f"{name}: exists={p.exists()}  size={p.stat().st_size if p.exists() else 0} bytes")
print("Top-level entries:", sorted(os.listdir(DATA_DIR))[:20])


## Write the training script

Identical pipeline to the Kaggle notebook, with four small A100/Colab adjustments baked in:
- `/content/...` paths added to the data-root search list
- `Accelerator(mixed_precision="bf16", ...)`
- Base model loaded with `torch_dtype=torch.bfloat16`
- TF32 enabled (`torch.backends.cuda.matmul.allow_tf32 = True`)

Everything else — LoRA config, hard-negative mining, the verifier prompt, eval loop — is unchanged.


In [ ]:
from pathlib import Path
training_script = r"""import argparse
import gc
import json
import math
import os
import random
import re
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from accelerate import Accelerator
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForImageTextToText, AutoProcessor, get_linear_schedule_with_warmup


VERIFIER_INSTRUCTION = (
    "Decide whether the candidate answer option is correct for the science question. "
    "Reply with exactly one word: yes or no."
)


def get_cfg() -> argparse.Namespace:
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset-root", type=str, default="")
    ap.add_argument("--output-dir", type=str, default="./run_14_outputs")
    ap.add_argument("--model-id", type=str, default="HuggingFaceTB/SmolVLM-500M-Instruct")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--epochs", type=int, default=4)
    ap.add_argument("--train-batch-size", type=int, default=1)
    ap.add_argument("--eval-batch-size", type=int, default=1)
    ap.add_argument("--grad-accum-steps", type=int, default=8)
    ap.add_argument("--learning-rate", type=float, default=2e-4)
    ap.add_argument("--weight-decay", type=float, default=0.01)
    ap.add_argument("--warmup-ratio", type=float, default=0.05)
    ap.add_argument("--max-grad-norm", type=float, default=1.0)
    ap.add_argument("--logging-steps", type=int, default=25)
    ap.add_argument("--max-image-longest-edge", type=int, default=640)
    ap.add_argument("--max-hint-chars", type=int, default=350)
    ap.add_argument("--max-lecture-chars", type=int, default=500)
    ap.add_argument("--checkpoint-eval-size", type=int, default=256)
    ap.add_argument("--early-stopping-patience", type=int, default=2)
    ap.add_argument("--final-full-eval-top-k", type=int, default=2)
    ap.add_argument("--lora-r", type=int, default=16)
    ap.add_argument("--lora-alpha", type=int, default=32)
    ap.add_argument("--lora-dropout", type=float, default=0.05)
    ap.add_argument("--hard-negatives-per-question", type=int, default=2)
    return ap.parse_args()


def fix_randomness(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_data_root(root_hint: str) -> Path:
    search_paths = []
    if root_hint:
        search_paths.append(Path(root_hint))
    search_paths.extend(
        [
            Path.cwd(),
            Path.cwd() / "pixels-to-predictions",
            # Google Colab default locations
            Path("/content/pixels-to-predictions"),
            Path("/content/data/pixels-to-predictions"),
            Path("/content"),
            Path("/content/data"),
            Path("/content/drive/MyDrive/pixels-to-predictions"),
            # Kaggle locations (kept so the same script still works there)
            Path("/kaggle/input/pixels-to-predictions"),
            Path("/kaggle/input"),
            Path("/kaggle/input/competitions/pixels-to-predictions"),
        ]
    )
    for p in search_paths:
        if (p / "train.csv").exists() and (p / "val.csv").exists() and (p / "test.csv").exists():
            return p
    raise FileNotFoundError("Could not locate train.csv/val.csv/test.csv.")


def find_image_root(data_root: Path) -> Path:
    for p in [data_root, data_root / "images"]:
        if (p / "images" / "train").exists() and (p / "images" / "val").exists():
            return p
    raise FileNotFoundError("Could not locate image root.")


def truncate_field(raw: Optional[str], limit: int) -> str:
    if not isinstance(raw, str):
        return ""
    normalized = re.sub(r"\s+", " ", raw).strip()
    if len(normalized) <= limit:
        return normalized
    return normalized[:limit].rsplit(" ", 1)[0] + " ..."


def word_tokens(txt: str) -> set[str]:
    if not isinstance(txt, str):
        return set()
    return set(re.findall(r"[a-z0-9]+", txt.lower()))


def jaccard_sim(a: str, b: str) -> float:
    tok_a = word_tokens(a)
    tok_b = word_tokens(b)
    if not tok_a or not tok_b:
        return 0.0
    return len(tok_a & tok_b) / len(tok_a | tok_b)


def make_verifier_prompt(entry: pd.Series, opt_idx: int, hint_limit: int, lecture_limit: int) -> str:
    opt_text = entry["choices_list"][opt_idx]
    parts = [
        VERIFIER_INSTRUCTION,
        f"Question: {truncate_field(entry['question'], 500)}",
        f"Candidate option ({opt_idx}): {opt_text}",
    ]
    hint_str = truncate_field(entry.get("hint", ""), hint_limit)
    lec_str = truncate_field(entry.get("lecture", ""), lecture_limit)
    if hint_str:
        parts.append(f"Hint: {hint_str}")
    if lec_str:
        parts.append(f"Lecture: {lec_str}")
    parts.append("Is this candidate option correct? Answer yes or no.")
    return "\n\n".join(parts)


def load_splits(data_root: Path, img_root: Path, cfg: argparse.Namespace):
    tr = pd.read_csv(data_root / "train.csv")
    va = pd.read_csv(data_root / "val.csv")
    te = pd.read_csv(data_root / "test.csv")

    for frame, tag in [(tr, "train"), (va, "val"), (te, "test")]:
        frame["split"] = tag
        frame["choices_list"] = frame["choices"].apply(json.loads)
        frame["abs_img_path"] = frame["image_path"].apply(lambda x: img_root / x)

    va["stratify_key"] = va["answer"].astype(str) + "__" + va["num_choices"].astype(str)
    frac = min(1.0, cfg.checkpoint_eval_size / len(va))
    buckets = []
    for _, grp in va.groupby("stratify_key", sort=False):
        n = max(1, round(len(grp) * frac))
        buckets.append(grp.sample(n=min(n, len(grp)), random_state=cfg.seed))
    quick_eval = pd.concat(buckets, axis=0).drop_duplicates(subset=["id"])
    if len(quick_eval) > cfg.checkpoint_eval_size:
        quick_eval = quick_eval.sample(cfg.checkpoint_eval_size, random_state=cfg.seed)
    quick_eval = quick_eval.reset_index(drop=True)

    return tr, va, te, quick_eval


def expand_to_option_rows(frame: pd.DataFrame, cfg: argparse.Namespace, with_labels: bool) -> List[Dict]:
    items: List[Dict] = []
    for qi, qrow in enumerate(frame.to_dict("records")):
        entry = pd.Series(qrow)
        for oi in range(int(qrow["num_choices"])):
            lbl = None
            if with_labels:
                lbl = "yes" if int(qrow["answer"]) == oi else "no"
            items.append(
                {
                    "q_idx": qi,
                    "opt_idx": oi,
                    "n_opts": int(qrow["num_choices"]),
                    "gt_answer": int(qrow["answer"]) if with_labels else -1,
                    "img_path": qrow["abs_img_path"],
                    "prompt_text": make_verifier_prompt(
                        entry,
                        opt_idx=oi,
                        hint_limit=cfg.max_hint_chars,
                        lecture_limit=cfg.max_lecture_chars,
                    ),
                    "target_word": lbl,
                }
            )
    return items


def balance_by_repeating(pool: List[Dict], goal: int, seed: int, tag: str) -> List[Dict]:
    if not pool or len(pool) >= goal:
        return [r.copy() for r in pool[:goal]]
    rng = random.Random(seed)
    out: List[Dict] = []
    for i in range(goal):
        item = pool[i % len(pool)].copy()
        item[f"{tag}_dup_id"] = i
        out.append(item)
    rng.shuffle(out)
    return out


def build_training_pool(frame: pd.DataFrame, cfg: argparse.Namespace) -> List[Dict]:
    pos_pool: List[Dict] = []
    neg_pool: List[Dict] = []

    for qi, qrow in enumerate(frame.to_dict("records")):
        entry = pd.Series(qrow)
        correct_oi = int(qrow["answer"])
        correct_txt = qrow["choices_list"][correct_oi]
        q_txt = qrow["question"]
        h_txt = qrow.get("hint", "")
        l_txt = qrow.get("lecture", "")
        context_anchor = " ".join(
            s for s in [q_txt, truncate_field(h_txt, cfg.max_hint_chars), truncate_field(l_txt, 200)] if s
        )

        pos_pool.append(
            {
                "q_idx": qi,
                "opt_idx": correct_oi,
                "n_opts": int(qrow["num_choices"]),
                "gt_answer": correct_oi,
                "img_path": qrow["abs_img_path"],
                "prompt_text": make_verifier_prompt(
                    entry,
                    opt_idx=correct_oi,
                    hint_limit=cfg.max_hint_chars,
                    lecture_limit=cfg.max_lecture_chars,
                ),
                "target_word": "yes",
            }
        )

        distractor_pool: List[Dict] = []
        for oi in range(int(qrow["num_choices"])):
            if oi == correct_oi:
                continue
            opt_txt = qrow["choices_list"][oi]
            sim_to_correct = jaccard_sim(opt_txt, correct_txt)
            sim_to_context = jaccard_sim(opt_txt, context_anchor)
            n_toks = len(word_tokens(opt_txt))
            difficulty = (
                0.6 * sim_to_correct
                + 0.25 * sim_to_context
                + 0.05 * min(n_toks / 12.0, 1.0)
                + 0.10 * max(int(qrow["num_choices"]) - 2, 0)
            )
            distractor_pool.append(
                {
                    "q_idx": qi,
                    "opt_idx": oi,
                    "n_opts": int(qrow["num_choices"]),
                    "gt_answer": correct_oi,
                    "img_path": qrow["abs_img_path"],
                    "prompt_text": make_verifier_prompt(
                        entry,
                        opt_idx=oi,
                        hint_limit=cfg.max_hint_chars,
                        lecture_limit=cfg.max_lecture_chars,
                    ),
                    "target_word": "no",
                    "difficulty": difficulty,
                }
            )

        distractor_pool.sort(key=lambda r: r["difficulty"], reverse=True)
        take = min(cfg.hard_negatives_per_question, len(distractor_pool))
        neg_pool.extend(distractor_pool[:take])

    if not pos_pool or not neg_pool:
        return pos_pool + neg_pool

    n_target = max(len(pos_pool), len(neg_pool))
    pos_balanced = balance_by_repeating(pos_pool, goal=n_target, seed=cfg.seed, tag="yes")
    neg_balanced = balance_by_repeating(neg_pool, goal=n_target, seed=cfg.seed + 1, tag="no")

    combined = pos_balanced + neg_balanced
    random.Random(cfg.seed).shuffle(combined)
    return combined


class FinetuneDataset(Dataset):
    def __init__(self, samples: List[Dict]):
        self.samples = samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, i: int) -> Dict:
        return self.samples[i]


class InferenceDataset(Dataset):
    def __init__(self, samples: List[Dict]):
        self.samples = samples

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, i: int) -> Dict:
        return self.samples[i]


class SupervisedBatchBuilder:
    def __init__(self, tok: AutoProcessor):
        self.tok = tok

    def __call__(self, batch_items: List[Dict]) -> Dict[str, torch.Tensor]:
        ids_list, mask_list, pix_list, pixmask_list, lbl_list = [], [], [], [], []

        for item in batch_items:
            user_turn = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "path": str(item["img_path"])},
                        {"type": "text", "text": item["prompt_text"]},
                    ],
                }
            ]
            full_convo = user_turn + [
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": item["target_word"]}],
                }
            ]

            prompt_enc = self.tok.apply_chat_template(
                user_turn,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                add_generation_prompt=True,
            )
            full_enc = self.tok.apply_chat_template(
                full_convo,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )

            tok_ids = full_enc["input_ids"][0]
            attn = full_enc["attention_mask"][0]
            lbls = tok_ids.clone()
            lbls[: prompt_enc["input_ids"].shape[1]] = -100
            lbls[attn == 0] = -100

            ids_list.append(tok_ids)
            mask_list.append(attn)
            pix_list.append(full_enc["pixel_values"][0])
            pixmask_list.append(full_enc["pixel_attention_mask"][0])
            lbl_list.append(lbls)

        pad_id = self.tok.tokenizer.pad_token_id
        return {
            "input_ids": torch.nn.utils.rnn.pad_sequence(ids_list, batch_first=True, padding_value=pad_id),
            "attention_mask": torch.nn.utils.rnn.pad_sequence(mask_list, batch_first=True, padding_value=0),
            "pixel_values": torch.stack(pix_list),
            "pixel_attention_mask": torch.stack(pixmask_list),
            "labels": torch.nn.utils.rnn.pad_sequence(lbl_list, batch_first=True, padding_value=-100),
        }


class InferenceBatchBuilder:
    def __init__(self, tok: AutoProcessor):
        self.tok = tok

    def __call__(self, batch_items: List[Dict]) -> Dict[str, torch.Tensor]:
        ids_list, mask_list, pix_list, pixmask_list = [], [], [], []
        q_idxs, o_idxs, n_opts_list, gt_list = [], [], [], []

        for item in batch_items:
            convo = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "path": str(item["img_path"])},
                        {"type": "text", "text": item["prompt_text"]},
                    ],
                }
            ]
            enc = self.tok.apply_chat_template(
                convo,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                add_generation_prompt=True,
            )

            ids_list.append(enc["input_ids"][0])
            mask_list.append(enc["attention_mask"][0])
            pix_list.append(enc["pixel_values"][0])
            pixmask_list.append(enc["pixel_attention_mask"][0])
            q_idxs.append(int(item["q_idx"]))
            o_idxs.append(int(item["opt_idx"]))
            n_opts_list.append(int(item["n_opts"]))
            gt_list.append(int(item["gt_answer"]))

        pad_id = self.tok.tokenizer.pad_token_id
        return {
            "input_ids": torch.nn.utils.rnn.pad_sequence(ids_list, batch_first=True, padding_value=pad_id),
            "attention_mask": torch.nn.utils.rnn.pad_sequence(mask_list, batch_first=True, padding_value=0),
            "pixel_values": torch.stack(pix_list),
            "pixel_attention_mask": torch.stack(pixmask_list),
            "q_idxs": torch.tensor(q_idxs, dtype=torch.long),
            "o_idxs": torch.tensor(o_idxs, dtype=torch.long),
            "n_opts": torch.tensor(n_opts_list, dtype=torch.long),
            "gt_answers": torch.tensor(gt_list, dtype=torch.long),
        }


def resolve_single_token_id(tok: AutoProcessor, word: str) -> int:
    ids = tok.tokenizer.encode(word, add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"Word '{word}' tokenizes to {ids}, expected exactly 1 token.")
    return ids[0]


def snapshot_adapter_weights(net: torch.nn.Module) -> Dict[str, torch.Tensor]:
    return {n: p.detach().cpu().clone() for n, p in net.named_parameters() if p.requires_grad}


def restore_adapter_weights(net: torch.nn.Module, snapshot: Dict[str, torch.Tensor]) -> None:
    param_map = dict(net.named_parameters())
    for n, t in snapshot.items():
        param_map[n].data.copy_(t.to(device=param_map[n].device, dtype=param_map[n].dtype))


@torch.inference_mode()
def run_scoring(
    accel: Accelerator,
    net: torch.nn.Module,
    tok: AutoProcessor,
    eval_frame: pd.DataFrame,
    cfg: argparse.Namespace,
    yes_id: int,
    no_id: int,
) -> Dict[str, List[int]]:
    option_rows = expand_to_option_rows(eval_frame.reset_index(drop=True), cfg=cfg, with_labels="answer" in eval_frame.columns)
    infer_ds = InferenceDataset(option_rows)
    infer_dl = DataLoader(
        infer_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        collate_fn=InferenceBatchBuilder(tok),
        num_workers=2,
        pin_memory=True,
    )

    all_q: List[int] = []
    all_o: List[int] = []
    all_margin: List[float] = []
    all_gt: List[int] = []

    net.eval()
    for mb in infer_dl:
        mb = {k: v.to(accel.device) for k, v in mb.items()}
        q_idx_t = mb.pop("q_idxs")
        o_idx_t = mb.pop("o_idxs")
        gt_t = mb.pop("gt_answers")
        mb.pop("n_opts")

        fwd = net(**mb)
        last_logits = fwd.logits[:, -1, :]
        margin = last_logits[:, yes_id] - last_logits[:, no_id]

        all_q.extend(q_idx_t.cpu().tolist())
        all_o.extend(o_idx_t.cpu().tolist())
        all_margin.extend(margin.cpu().tolist())
        all_gt.extend(gt_t.cpu().tolist())

    q_to_opts: Dict[int, List[Dict]] = {}
    q_to_gt: Dict[int, int] = {}
    for qi, oi, sc, gt in zip(all_q, all_o, all_margin, all_gt):
        q_to_opts.setdefault(qi, []).append({"oi": oi, "sc": sc})
        q_to_gt[qi] = gt

    predictions, ground_truths = [], []
    for qi in sorted(q_to_opts):
        winner = max(q_to_opts[qi], key=lambda x: x["sc"])
        predictions.append(int(winner["oi"]))
        ground_truths.append(int(q_to_gt[qi]))

    return {"preds": predictions, "answers": ground_truths}


def run() -> None:
    cfg = get_cfg()
    fix_randomness(cfg.seed)
    # A100-friendly defaults: enable TF32 for fp32 matmul/cudnn paths
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

    accel = Accelerator(mixed_precision="bf16", gradient_accumulation_steps=cfg.grad_accum_steps)
    save_dir = Path(cfg.output_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    data_root = find_data_root(cfg.dataset_root)
    img_root = find_image_root(data_root)
    tr_frame, va_frame, te_frame, quick_eval_frame = load_splits(data_root, img_root, cfg)

    print(
        {
            "num_processes": accel.num_processes,
            "data_root": str(data_root),
            "img_root": str(img_root),
            "train_rows": len(tr_frame),
            "val_rows": len(va_frame),
            "quick_eval_rows": len(quick_eval_frame),
        }
    )

    tok = AutoProcessor.from_pretrained(cfg.model_id, size={"longest_edge": cfg.max_image_longest_edge})
    tok.tokenizer.padding_side = "right"
    yes_id = resolve_single_token_id(tok, "yes")
    no_id = resolve_single_token_id(tok, "no")

    net = AutoModelForImageTextToText.from_pretrained(
        cfg.model_id,
        torch_dtype=torch.bfloat16,
        _attn_implementation="sdpa",
    )
    net.config.use_cache = False
    net.gradient_checkpointing_enable()

    adapter_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    net = get_peft_model(net, adapter_cfg)

    n_trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in net.parameters())
    print(
        {
            "trainable_params": n_trainable,
            "total_params": n_total,
            "trainable_pct": round(100 * n_trainable / n_total, 4),
        }
    )
    assert n_trainable < 5_000_000, "Trainable parameter budget exceeded."

    tr_samples = build_training_pool(tr_frame, cfg=cfg)
    n_yes = sum(1 for s in tr_samples if s["target_word"] == "yes")
    n_no = sum(1 for s in tr_samples if s["target_word"] == "no")
    print(
        {
            "train_samples": len(tr_samples),
            "yes_samples": n_yes,
            "no_samples": n_no,
            "hard_negatives_per_question": cfg.hard_negatives_per_question,
        }
    )

    tr_dl = DataLoader(
        FinetuneDataset(tr_samples),
        batch_size=cfg.train_batch_size,
        shuffle=True,
        collate_fn=SupervisedBatchBuilder(tok),
        num_workers=2,
        pin_memory=True,
    )

    opt = torch.optim.AdamW(
        [p for p in net.parameters() if p.requires_grad],
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    updates_per_epoch = math.ceil(len(tr_dl) / cfg.grad_accum_steps)
    total_updates = updates_per_epoch * cfg.epochs
    n_warmup = int(total_updates * cfg.warmup_ratio)
    sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=n_warmup, num_training_steps=total_updates)

    net, opt, tr_dl, sched = accel.prepare(net, opt, tr_dl, sched)

    peak_quick_acc = -1.0
    patience_counter = 0
    saved_checkpoints: List[Dict] = []

    for ep in range(1, cfg.epochs + 1):
        net.train()
        cumulative_loss = 0.0

        for step_i, mb in enumerate(tr_dl, start=1):
            with accel.accumulate(net):
                out = net(**mb)
                loss_val = out.loss
                accel.backward(loss_val)
                if accel.sync_gradients:
                    accel.clip_grad_norm_(net.parameters(), cfg.max_grad_norm)
                opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True)

            cumulative_loss += loss_val.detach().item()
            if step_i % cfg.logging_steps == 0:
                print(f"epoch={ep} step={step_i}/{len(tr_dl)} avg_loss={cumulative_loss / step_i:.4f}")

        quick_out = run_scoring(
            accel=accel,
            net=net,
            tok=tok,
            eval_frame=quick_eval_frame,
            cfg=cfg,
            yes_id=yes_id,
            no_id=no_id,
        )

        quick_acc = accuracy_score(quick_out["answers"], quick_out["preds"])
        print(f"checkpoint_validation_accuracy_full={quick_acc:.4f}")

        if quick_acc > peak_quick_acc:
            peak_quick_acc = quick_acc
            patience_counter = 0
            ckpt_path = save_dir / f"candidate_epoch_{ep}.pt"
            torch.save(snapshot_adapter_weights(accel.unwrap_model(net)), ckpt_path)
            saved_checkpoints.append({"epoch": ep, "subset_score": quick_acc, "path": str(ckpt_path)})
            saved_checkpoints = sorted(saved_checkpoints, key=lambda x: x["subset_score"], reverse=True)[: cfg.final_full_eval_top_k]
            print(f"saved_candidate_state epoch={ep} subset_score={quick_acc:.4f}")
        else:
            patience_counter += 1
            print(f"no_improvement_count={patience_counter}/{cfg.early_stopping_patience}")

        if patience_counter >= cfg.early_stopping_patience:
            print("early_stopping_triggered=True")
            break

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    manifest = save_dir / "candidate_manifest.json"
    manifest.write_text(json.dumps(saved_checkpoints, indent=2), encoding="utf-8")

    best_val_acc = -1.0
    winner_path: Optional[str] = None
    winner_epoch: Optional[int] = None

    for ckpt in saved_checkpoints:
        weights = torch.load(ckpt["path"], map_location="cpu")
        restore_adapter_weights(accel.unwrap_model(net), weights)

        full_out = run_scoring(
            accel=accel,
            net=net,
            tok=tok,
            eval_frame=va_frame,
            cfg=cfg,
            yes_id=yes_id,
            no_id=no_id,
        )

        full_acc = accuracy_score(full_out["answers"], full_out["preds"])
        print(
            f"full_validation_accuracy_epoch_{ckpt['epoch']}={full_acc:.4f} "
            f"(subset_score={ckpt['subset_score']:.4f})"
        )
        if full_acc > best_val_acc:
            best_val_acc = full_acc
            winner_path = ckpt["path"]
            winner_epoch = ckpt["epoch"]

    winner_path = winner_path or saved_checkpoints[0]["path"]
    restore_adapter_weights(accel.unwrap_model(net), torch.load(winner_path, map_location="cpu"))

    adapter_out = save_dir / "best_adapter"
    adapter_out.mkdir(parents=True, exist_ok=True)
    accel.unwrap_model(net).save_pretrained(adapter_out, save_function=accel.save)

    summary = {
        "best_checkpoint_subset_score": peak_quick_acc,
        "best_full_validation_score": best_val_acc,
        "best_epoch": winner_epoch,
        "num_processes": accel.num_processes,
        "pipeline": "binary_choice_scoring_yes_no_hard_negative_balance",
        "hard_negatives_per_question": cfg.hard_negatives_per_question,
    }
    (save_dir / "metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(summary)

    test_out = run_scoring(
        accel=accel,
        net=net,
        tok=tok,
        eval_frame=te_frame,
        cfg=cfg,
        yes_id=yes_id,
        no_id=no_id,
    )

    sub = te_frame[["id"]].copy().reset_index(drop=True)
    sub["answer"] = test_out["preds"]
    sub_path = save_dir / "submission.csv"
    sub.to_csv(sub_path, index=False)
    print(f"saved_submission={sub_path}")


if __name__ == "__main__":
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    run()
"""
out_path = Path("/content/train_single_gpu.py")
out_path.write_text(training_script, encoding="utf-8")
print("wrote", out_path, "(", out_path.stat().st_size, "bytes )")


## Important Notes

- Make sure you selected an **A100 GPU** runtime before running training.
- If you restart the runtime, rerun all previous cells.
- The dataset will be downloaded into:
  `/content/pixels-to-predictions`
- Training outputs will be saved in:
  `/content/run_14_outputs`


## Launch training

Hyperparameters bumped for A100 40 GB:

| flag | Kaggle (T4/P100) | This notebook (A100) |
| --- | --- | --- |
| `--mixed_precision` | fp16 | **bf16** |
| `--train-batch-size` | 1 | **4** |
| `--grad-accum-steps` | 8 | **2** |
| `--eval-batch-size` | 1 | **8** |

Effective training batch size stays at 8 (4 × 2), so the optimization trajectory is the same — you just spend ~4× less wall clock per step. If you have an 80 GB A100, you can push `--train-batch-size 8 --grad-accum-steps 1` and `--eval-batch-size 16` for another ~2× speedup.


In [ ]:
# Launch training on a single A100 GPU

!accelerate launch     --mixed_precision=bf16     /content/train_single_gpu.py     --dataset-root /content/pixels-to-predictions     --output-dir /content/run_14_outputs     --train-batch-size 4     --grad-accum-steps 2     --eval-batch-size 8


In [ ]:
!ls -R /content/run_14_outputs


## (Optional) Save outputs to Google Drive

Colab's `/content` is wiped when the runtime stops. Run the next cell to copy the run directory to Drive so you keep the LoRA adapter, the submission CSV, and the metrics.


In [ ]:
# Mount Drive and copy outputs. Comment this out if you don't want it.
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive", force_remount=False)

src = Path("/content/run_14_outputs")
dst = Path("/content/drive/MyDrive/pixels_to_predictions/run_14_outputs")
dst.parent.mkdir(parents=True, exist_ok=True)
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print("copied", src, "->", dst)
